# Notebook 1 / 2 — Reproduce TabPFN v2 (Hollmann et al., Nature 2025)

**Kaggle:** Accelerator = **GPU T4 x2**, Internet = **On**. Thời gian ước tính: ~1–1,5 giờ.

Kiểm chứng kết luận chính của paper: dùng ngay không cần tune, TabPFN v2 tốt hơn các mô hình
cổ điển (tham số mặc định) trên dữ liệu bảng nhỏ.

* Dữ liệu: OpenML-CC18 (suite 99), lọc ≤ 3000 dòng, ≤ 100 đặc trưng, ≤ 10 lớp (33 dataset).
* Giao thức: stratified 3-fold CV, cùng fold cho mọi phương pháp.
* Phương pháp: TabPFN v2 (checkpoint gốc của paper) vs Logistic Regression, kNN, Random Forest,
  XGBoost, LightGBM, CatBoost.
* Metric: ROC-AUC (chính), accuracy, log-loss, mean rank, Wilcoxon.

`--val_preds` lưu thêm dự đoán trên tập validation nội bộ, Notebook 2 dùng lại (không phải chạy lại).
Script chạy tiếp được nếu bị ngắt: chạy lại cell, phần đã xong sẽ được bỏ qua.

## 0. Cài đặt chung

In [ ]:
# >>>>> SỬA DÒNG NÀY: URL repo GitHub của bạn <<<<<
REPO_URL = "https://github.com/<YOUR_GITHUB_USERNAME>/tabpfn-dl-project.git"

import os
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
REPO = os.path.join(WORK, "tabpfn-dl-project")
os.environ["OPENML_CACHE"] = os.path.join(WORK, "openml_cache")
print(WORK, REPO)

In [ ]:
!nvidia-smi -L
if not os.path.isdir(REPO):
    !git clone -q {REPO_URL} {REPO}
else:
    !cd {REPO} && git pull -q
%cd {REPO}
!pip install -q -r requirements.txt
!python tests/test_smoke.py

In [ ]:
import torch, tabpfn, subprocess, time, sys, glob, shutil, zipfile
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(),
      "| GPUs", torch.cuda.device_count(), "| tabpfn", tabpfn.__version__)
os.makedirs("logs", exist_ok=True)

def run(cmd, log):
    """Chạy lệnh shell, in log ra cell và ghi vào file."""
    print(">>", cmd)
    with open(log, "a") as f:
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            sys.stdout.write(line); f.write(line)
        p.wait()
    if p.returncode != 0:
        print(f"!! exit code {p.returncode} (xem {log})")

def run_parallel(jobs):
    """jobs = [(cmd, gpu_id, logfile), ...] -> mỗi GPU một tiến trình, chờ tất cả."""
    procs = []
    for cmd, gpu, log in jobs:
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}
        print(f">> [GPU {gpu}] {cmd}")
        procs.append((subprocess.Popen(cmd, shell=True, env=env, stdout=open(log, "a"),
                                       stderr=subprocess.STDOUT), log))
    while any(p.poll() is None for p, _ in procs):
        time.sleep(120)
        for _, log in procs:
            lines = [l for l in open(log).read().splitlines() if l.strip()]
            print(f"[{log}] {lines[-1] if lines else ''}")
    print("all jobs finished")

## 1. Phần A — Reproduce

In [ ]:
BENCH = "results/benchmark"
common = f"--source openml --max_rows 3000 --n_folds 3 --out {BENCH}"
run(f"python scripts/run_benchmark.py {common} --val_preds "
    f"--methods logreg,knn,rf,xgb,lgbm,catboost,tabpfn_v2", "logs/partA.log")

## 2. Báo cáo Phần A + đóng gói

Bảng gồm: kết quả tổng hợp, mean rank, win/tie/loss và p-value so với TabPFN v2, danh sách
dataset mà TabPFN v2 thua (điểm yếu mà Notebook 2 nhắm vào).

In [ ]:
!python scripts/make_report.py --bench {BENCH} --out results/report
!cd {REPO} && zip -qr {WORK}/session1_results.zip results logs
from IPython.display import Markdown, display, Image
display(Markdown(open("results/report/report.md").read()))
for f in sorted(glob.glob("results/report/fig_A*.png")):
    display(Image(f))
print("-> TẢI VỀ:", f"{WORK}/session1_results.zip  (dùng cho Notebook 2)")